# Week 5: First Qiskit Program on Real IBM Quantum Hardware
## Bell State Experiment — From Simulation to Real QPU

**Objective:** In this assignment, we will understand the quantum computing workflow using the Qiskit Pattern (Map → Optimize → Execute → Post-process), run our first quantum circuit (a Bell state) on actual IBM quantum hardware, and learn the importance of reproducibility logging in quantum experiments.

### Section 1: Environment Setup & IBM Quantum Authentication
First, we need to ensure the necessary packages are installed (`qiskit`, `qiskit-ibm-runtime`, `qiskit-aer`, `matplotlib`). Then, we authenticate with the IBM Quantum service and select an appropriate backend.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# STEP 0: Save your IBM Quantum credentials (run once) ===
QiskitRuntimeService.save_account(
     channel="ibm_quantum_platform",
     token="",
     overwrite=True
 )

# Load saved credentials
service = QiskitRuntimeService(channel="ibm_quantum_platform")
print(f"Connected to IBM Quantum")
print(f"Available backends: {[b.name for b in service.backends()]}")

# Select backend
backend = service.least_busy(simulator=False, operational=True)
print(f"Selected backend: {backend.name}")
print(f"Number of qubits: {backend.num_qubits}")
print(f"Basis gates: {backend.operation_names}")

### Section 2: STEP 1 — MAP: Build the Bell State Circuit
The Bell state $|\Phi^+\rangle$ is a maximally entangled state of two qubits, represented mathematically as:
$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

To create this state, we start with two qubits in the $|00\rangle$ state and apply the following gates:
1. **Hadamard Gate (H)** on the first qubit: creates a superposition $\frac{|0\rangle+|1\rangle}{\sqrt{2}} \otimes |0\rangle$
2. **CNOT Gate** with the first qubit as control and the second as target: entangles the qubits, resulting in $\frac{|00\rangle+|11\rangle}{\sqrt{2}}$

In [ ]:
from qiskit import QuantumCircuit

bell = QuantumCircuit(2)
bell.h(0)       # Hadamard: creates superposition
bell.cx(0, 1)   # CNOT: entangles qubits
bell.measure_all()

print('=== Abstract Bell Circuit ===')
print(f'Number of qubits: {bell.num_qubits}')
print(f'Gate operations: {bell.count_ops()}')
print(f'Circuit depth: {bell.depth()}')
bell.draw('mpl')

### Section 3: STEP 2 — OPTIMIZE: Transpile to ISA Circuit
Abstract quantum circuits must be transformed to match the target quantum hardware's Instruction Set Architecture (ISA). This process, called **transpilation**, translates the ideal gates into the backend's specific basis gates and maps virtual qubits to physical qubits according to the device's coupling map.

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
bell_isa = pm.run(bell)

print('=== Before Transpilation (Abstract) ===')
print(f'  Gates: {bell.count_ops()}')
print(f'  Depth: {bell.depth()}')
print()
print('=== After Transpilation (ISA) ===')
print(f'  Gates: {bell_isa.count_ops()}')
print(f'  Depth: {bell_isa.depth()}')
print(f'  Physical qubits used: {bell_isa.layout.final_index_layout(filter_ancillas=True)}')

bell_isa.draw('mpl', fold=-1)

### Section 4: STEP 3 — EXECUTE: Run on Real IBM Quantum Hardware
Now we submit the ISA circuit to the selected real quantum hardware. Real quantum computers are susceptible to noise (decoherence, gate errors, readout errors), so the results will not be as perfect as in a simulator. We use `SamplerV2` to gather sampling statistics over many shots.

In [ ]:
from qiskit_ibm_runtime import SamplerV2
import time

sampler = SamplerV2(backend)
print(f'Submitting job to {backend.name}...')
start_time = time.time()
job = sampler.run([bell_isa], shots=4096)
print(f'Job ID: {job.job_id()}')
print(f'Job status: {job.status()}')

# Wait for results
result = job.result()
elapsed = time.time() - start_time
print(f'\nJob completed in {elapsed:.1f} seconds')

counts_hw = result[0].data.meas.get_counts()
total_hw = sum(counts_hw.values())
print(f'\n=== Real Hardware Results ===')
print(f'Total shots: {total_hw}')
print(f'Counts: {counts_hw}')
print()
print('Probabilities:')
for bs in sorted(counts_hw.keys()):
    prob = counts_hw[bs] / total_hw
    print(f'  |{bs}\u27e9: {counts_hw[bs]}/{total_hw} = {prob:.4f}')

### Section 5: Simulator Comparison (Baseline)
To quantify the noise in our real hardware execution, we run the same circuit on a noiseless simulator (`AerSimulator`). This serves as our baseline expectation.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSamplerV2

sim_backend = AerSimulator()
pm_sim = generate_preset_pass_manager(target=sim_backend.target, optimization_level=3)
bell_sim_isa = pm_sim.run(bell)

sampler_sim = BackendSamplerV2(backend=sim_backend)
job_sim = sampler_sim.run([bell_sim_isa], shots=4096)
result_sim = job_sim.result()
counts_sim = result_sim[0].data.meas.get_counts()
total_sim = sum(counts_sim.values())

print('=== Simulator Results ===')
print(f'Counts: {counts_sim}')
for bs in sorted(counts_sim.keys()):
    print(f'  |{bs}\u27e9: {counts_sim[bs]}/{total_sim} = {counts_sim[bs]/total_sim:.4f}')

### Section 6: STEP 4 — POST-PROCESS: Visualize & Compare
In the post-processing phase, we visualize the distributions from both the ideal simulator and the noisy real hardware. We also calculate the fidelity metric, evaluating how closely our actual results match the desired outcome (measuring $|00\rangle$ or $|11\rangle$).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_states = ['00', '01', '10', '11']
probs_hw = [counts_hw.get(s, 0) / total_hw for s in all_states]
probs_sim = [counts_sim.get(s, 0) / total_sim for s in all_states]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simulator
colors_sim = ['#4CAF50' if s in ['00', '11'] else '#E0E0E0' for s in all_states]
axes[0].bar(all_states, probs_sim, color=colors_sim, edgecolor='black')
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Ideal 50%')
axes[0].set_title('Simulator (AerSimulator)', fontsize=13)
axes[0].set_xlabel('Bitstring')
axes[0].set_ylabel('Probability')
axes[0].set_ylim(0, 0.7)
axes[0].legend()
for i, (s, p) in enumerate(zip(all_states, probs_sim)):
    if p > 0.01:
        axes[0].text(i, p + 0.02, f'{p:.3f}', ha='center', fontsize=10, fontweight='bold')

# Real Hardware
colors_hw = ['#2196F3' if s in ['00', '11'] else '#FFCDD2' for s in all_states]
axes[1].bar(all_states, probs_hw, color=colors_hw, edgecolor='black')
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Ideal 50%')
axes[1].set_title(f'Real Hardware ({backend.name})', fontsize=13)
axes[1].set_xlabel('Bitstring')
axes[1].set_ylabel('Probability')
axes[1].set_ylim(0, 0.7)
axes[1].legend()
for i, (s, p) in enumerate(zip(all_states, probs_hw)):
    if p > 0.01:
        axes[1].text(i, p + 0.02, f'{p:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Bell State: Simulator vs Real Quantum Hardware', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Fidelity metrics
fidelity_sim = (counts_sim.get('00', 0) + counts_sim.get('11', 0)) / total_sim
fidelity_hw = (counts_hw.get('00', 0) + counts_hw.get('11', 0)) / total_hw
print(f'\nCorrelation (|00\u27e9 + |11\u27e9):')
print(f'  Simulator: {fidelity_sim:.4f}')
print(f'  Hardware:  {fidelity_hw:.4f}')
print(f'  Deviation: {abs(fidelity_sim - fidelity_hw):.4f}')

### Section 7: Reproducibility Metadata Extraction
Scientific research relies heavily on reproducibility. Because quantum hardware performance drifts over time, logging the exact execution context is critical. Run this cell to automatically generate your reproducibility metadata, then paste it into your `reproducibility_log.md`.

In [ ]:
import qiskit
import datetime
import platform
import sys

try:
    import qiskit_ibm_runtime
    runtime_version = qiskit_ibm_runtime.__version__
except:
    runtime_version = 'N/A'

try:
    import qiskit_aer
    aer_version = qiskit_aer.__version__
except:
    aer_version = 'N/A'

print('=' * 60)
print('       REPRODUCIBILITY LOG \u2014 METADATA')
print('=' * 60)
print()
print('--- Experiment Info ---')
print(f'  Title:       Bell State on IBM Quantum Hardware')
print(f'  Date/Time:   {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Objective:   Demonstrate Qiskit Pattern on real QPU')
print()
print('--- Software Environment ---')
print(f'  Python:              {sys.version}')
print(f'  Qiskit:              {qiskit.__version__}')
print(f'  qiskit-ibm-runtime:  {runtime_version}')
print(f'  qiskit-aer:          {aer_version}')
print(f'  NumPy:               {np.__version__}')
print(f'  Matplotlib:          {plt.matplotlib.__version__}')
print(f'  OS:                  {platform.system()} {platform.release()}')
print()
print('--- Backend / Hardware ---')
print(f'  Backend name:        {backend.name}')
print(f'  Number of qubits:    {backend.num_qubits}')
print(f'  Basis gates:         {backend.operation_names}')
print()
print('--- Circuit Details ---')
print(f'  Abstract gates:      {bell.count_ops()}')
print(f'  Abstract depth:      {bell.depth()}')
print(f'  ISA gates:           {bell_isa.count_ops()}')
print(f'  ISA depth:           {bell_isa.depth()}')
print(f'  Optimization level:  3')
print()
print('--- Execution Parameters ---')
print(f'  Primitive:           SamplerV2')
print(f'  Shots:               4096')
print(f'  Job ID:              {job.job_id()}')
print(f'  Execution time:      {elapsed:.1f}s')
print()
print('--- Results Summary ---')
print(f'  Hardware counts:     {counts_hw}')
print(f'  Hardware fidelity:   {fidelity_hw:.4f}')
print(f'  Simulator counts:    {counts_sim}')
print(f'  Simulator fidelity:  {fidelity_sim:.4f}')
print(f'  Fidelity deviation:  {abs(fidelity_sim - fidelity_hw):.4f}')
print()
print('=' * 60)
print('Copy the above into your reproducibility_log.md')
print('=' * 60)

### Section 8: Summary & Key Takeaways

#### The Qiskit Pattern Workflow
| Step | Description | Action Performed |
|------|-------------|------------------|
| **Map** | Formulate problem as a quantum circuit | Built the abstract Bell state circuit |
| **Optimize** | Transpile for target hardware | Translated to ISA using pass manager |
| **Execute** | Run on QPU | Submitted via `SamplerV2` to IBM Quantum |
| **Post-process**| Analyze results | Visualized distributions and calculated fidelity |

#### Observations
- **Noise in Hardware:** Unlike the ideal simulator which outputs exactly 50/50 for $|00\rangle$ and $|11\rangle$, the real quantum hardware exhibits noise. We observed small probabilities for states $|01\rangle$ and $|10\rangle$ which represent errors (e.g. readout errors or decoherence).
- **Reproducibility:** Capturing metadata (software versions, backend used, compile times) is essential in quantum experiments since both hardware performance and compiler optimizations change frequently.

**References:**
- IBM Quantum course, lecture2.pdf
- Qiskit Documentation: [Qiskit Pattern](https://docs.quantum.ibm.com/run)